In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [2]:
BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
#BASE_DIR = "../../Data/"

# SENTINEL + CANOPY DATA

In [4]:
SENTINEL_DATA_CSV        = BASE_DIR + "/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"
sentinel_raw_df = pd.read_csv(SENTINEL_DATA_CSV)
print(sentinel_raw_df.shape)
assert len(sentinel_raw_df["simard_height_m"].head())
assert len(sentinel_raw_df["tandemx_height_m"].head())

(8774, 34)


In [5]:
sentinel_raw_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg', 'capture_start',
       'capture_end', 'sentinel_time', 'Blue', 'Green', 'Red', 'RE1', 'RE2',
       'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE', 'cloud_threshold_used',
       'simard_height_m', 'tandemx_height_m'],
      dtype='object')

## Sentinel feature selection

In [7]:
sent_non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'sentinel_time',       # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'diameter',            # Allometric
    'height',               # Allometric
    'cloud_threshold_used',
    'start_date'
]

SENT_interact_cols = [
    # Indices (excluding MNDWI which isn't a biomass index)
    'EVI', 'NBR', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
    # Raw structural bands
    'NIR', 'SWIR1', 'SWIR2',
]

target = 'plant_AGB_kg'

sent_feature_cols = [c for c in sentinel_raw_df.columns if c not in sent_non_feature_cols]

### Feature selection with Simard canopy height

In [9]:
X_sent_t = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select TANDEMX
X_sent_t = X_sent_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_sent_t = X_sent_t.drop(columns=['simard_height_m'])

In [10]:
X_sent_t.shape

(3880, 21)

In [11]:
X_sent_t.columns

Index(['plot_id', 'species', 'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
       'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE', 'height'],
      dtype='object')

### Create interaction terms

In [13]:
X_sent_t = create_interact_terms(X_sent_t, SENT_interact_cols)

### Feature selection with Simard canopy height

In [15]:
X_sent_s = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select SIMARD
X_sent_s = X_sent_s.rename({'simard_height_m': 'height'}, axis=1)
X_sent_s = X_sent_s.drop(columns=['tandemx_height_m'])

### Create interaction terms

In [17]:
X_sent_s = create_interact_terms(X_sent_s, SENT_interact_cols)

### Create target variable

In [19]:
y_sent = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][target].copy()

## Sentinel test features

In [21]:
test_cv = 5

In [22]:
sent_struct_features = ['height']
species              = ['species']

sent_bands   = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
sent_indices = ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

sent_top_spectral  = ['EVI', 'NIR', 'NBR', 'SWIR1']
sent_redband       = ['RE1', 'RE2', 'RE3']
sent_interaction_terms = [c for c in X_sent_t.columns if c.startswith('height_x_')]

# Curated interactions matching curated spectral sets
sent_top_spectral_interactions = [f'height_x_{b}' for b in sent_top_spectral
                                  if f'height_x_{b}' in sentinel_raw_df.columns]
sent_redband_interactions      = [f'height_x_{b}' for b in sent_redband
                                  if f'height_x_{b}' in sentinel_raw_df.columns]

sent_features_list = [
    # Non-spectral baselines
    #struct_features,  # already tested in EMIT
    #species,          # already tested in EMIT
    sent_struct_features + species,

    # Spectral alone
    sent_bands,
    sent_indices,
    sent_top_spectral,
    sent_redband,

    # Spectral + height
    sent_bands        + sent_struct_features,
    sent_indices      + sent_struct_features,
    sent_top_spectral + sent_struct_features,
    sent_redband      + sent_struct_features,

    # Spectral + height + species
    sent_bands        + sent_struct_features + species,
    sent_top_spectral + sent_struct_features + species,

    # Spectral + height + interactions (curated where it matters)
    sent_bands        + sent_struct_features + sent_interaction_terms,
    sent_top_spectral + sent_struct_features + sent_top_spectral_interactions,
    sent_redband      + sent_struct_features + sent_redband_interactions,

    # Full specturm of features
    sent_bands        + sent_indices + sent_struct_features + sent_interaction_terms + species
]

## Sentiel. Create plot groups.

In [24]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_sent = None
if 'plot_id' in X_sent_t.columns:
    plot_groups_sent = X_sent_t['plot_id'].copy()
    X_sent_t = X_sent_t.drop(columns=['plot_id'])

if 'plot_id' in X_sent_s.columns:
    X_sent_s = X_sent_s.drop(columns=['plot_id'])

near_zero_sites_sent, high_agb_sites_sent,\
    near_zero_plots_sent, high_agb_plots_sent = \
        get_low_and_high_agb_plots(y_sent,
                                   plot_groups_sent)

grp_sentinel = GROUP_INFO(near_zero_sites_sent,
                          high_agb_sites_sent,
                          near_zero_plots_sent,
                          high_agb_plots_sent,
                          groups=plot_groups_sent,
                          cv=test_cv)

High-AGB threshold  : 104.49 kg
Near-zero threshold : 0.004134

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032
  Frenchman Caye_1          : log var = 0.000753
  Frenchman Caye_2          : log var = 0.000381
  Frenchman Caye_3          : log var = 0.000693
  Frenchman Caye_4          : log var = 0.001306
  Frenchman Caye_5          : log var = 0.001283
  Frenchman Caye_6          : log var = 0.000158
  Shipstern Lagoon_1        : log var = 0.001064
  Shipstern Lagoon_3        : log var = 0.000232
  Shipstern Lagoon_4        : log var = 0.000113
  Shipstern Lagoon_5        : log var = 0.000052
  Shipstern Lagoon_6        : log var = 0.000135

High-AGB plots:
  Channel Caye_1            : max AGB = 310.9 kg
  Channel Caye_2            : max AGB = 206.4 kg
  Channel Caye_3            : max AGB = 427.2 kg
  Channel Caye_4            : max AGB = 237.6 kg
  Channel Caye_5            : max AGB = 170.4 kg
  Channel C

In [25]:
assert plot_groups_sent is not None

# SENTINEL-2 EXPERIMENTS

In [27]:
test_cv = 5

In [28]:
LINEAR_FULL    = ["linear", "ridge", "lasso", "elasticnet"]
LINEAR_NO_OLS  = ["ridge", "lasso", "elasticnet"]

data_combos= [("SENTINEL + TANDEMX", X_sent_t, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL),
              ("SENTINEL + SIMARD" , X_sent_s, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL)
             ]
model_list = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids   = [False, True]
is_groups  = [True]

In [29]:
experiment_dict = {}
experiment_registry = []

In [ ]:
for (combo_name, X, y, features_list, grp, linear_variants), model_name, use_grid, use_groups in product(
    data_combos, model_list, is_grids, is_groups
):
    # Linear models do not benefit from grid search.
    if model_name == 'linear' and use_grid:
        continue
    
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model_name} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)

    #run_label = f"{combo_name}, {model}, grid_{'yes' if use_grid else 'no'}, GROUPED: {'yes' if use_groups else 'no'}"

    exp_result = run_experiment(X, y,
                                is_groups       = use_groups,
                                group_info      = grp,
                                features_list   = features_list,
                                model_type      = model_name,
                                linear_variants = linear_variants,
                                is_grid         = use_grid,
                                is_stratified   = False,
                                exp_registry    = experiment_registry,
                                exp_total_list  = experiment_dict,
                                dataset         = combo_name)


DATA: SENTINEL + TANDEMX | MODEL: linear | GRID: False | GROUPED: True

[1/60]

RUN: Dataset: SENTINEL + TANDEMX, Model: LINEAR REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height', 'species']
 Test R²     : 0.0750
 Test RMSE   : 4.44 kg
 Train R² (log scale): 0.2887
 Train R² (orig scale): -0.0237
 Train RMSE  : 18.79 kg
 Num rows    : 3100
 Num Features: 4

 Cross-validation ---
CV R² mean: 0.2165
CV R² std : 0.2039
CV scores : [-0.025  0.592  0.198  0.13   0.188]

Grouped Cross-validation ---
Grouped CV R² mean: 0.2605
Grouped CV R² std : 0.2064
Grouped CV scores : [-0.029  0.184  0.117 -0.03   0.39   0.535  0.624  0.213  0.272  0.329]
 EVALUATION: 

Test set:
  R²   : 0.075
  RMSE : 4.44 kg

 ✅ Test R² is positive (0.075)

Regular CrossValidation:
  Mean   : 0.217
  Std    : 0.204
  Scores : [-0.025  0.592  0.198  0.13   0.188]
 ✅ CV mean is positive (0.217)

Grouped CrossValidation:
  Big Creek_5          :  -0.029  ⚠️  fold contamination — Channel Caye_1 in same fold
 

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF ALL EXPERIMENTS

In [ ]:
#tab_df = tabulate_results(experiment_dict, top_n=None)
tab_df = tabulate_registry(experiment_registry)

## SUMMARY OF BEST ACCEPTABLE RESULTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(experiment_dict)
tab_df = tabulate_results(best_results, top_n=10)

## BEST ONE

In [ ]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)

# SAVE EXPERIMENT RESULTS

In [ ]:
import pickle

with open('sentinel_expr_dictionary.pkl', 'wb') as f:
    pickle.dump(experiment_dict, f)

with open('sentinel_expr_registry.pkl', 'wb') as f:
    pickle.dump(experiment_registry, f)

In [ ]:
loaded_dict = {}
with open('sentinel_expr_dictionary.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

## Verify loaded values

In [ ]:
%run Model_functions.ipynb
verify_dicts(global_experiment_list, loaded_dict)

## Check if the loaded values are printabled

### Print the first experiment.

In [ ]:
%run Model_functions.ipynb
key_list = list(loaded_dict.keys())
print_experiment(loaded_dict, key_list[0])

### Display the best experiments from the loaded results

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(loaded_dict)
tab_df = tabulate_results(best_results, top_n=10)

### Display experiment from a chosen row among the best list

In [ ]:
%run Model_functions.ipynb
print_experiment_from_row(tab_df.iloc[0])